# 04. Model Checking in the Wumpus World

Tiga notebook sebelumnya menyiapkan tiga hal yang berbeda, dan baru di sini
ketiganya bertemu.

| Notebook | Yang sudah dibangun | Cara kerjanya |
|---|---|---|
| 01 | Wumpus World dan arsitektur KB agent | agent menerima percept, TELL ke knowledge base, ASK aksi berikutnya |
| 02 | entailment $KB \vDash \alpha$ | $\alpha$ benar di setiap model yang membuat KB benar |
| 03 | logika proposisional | proposition symbol dirangkai dengan connective, nilainya dihitung lewat truth table |

Ada satu hal yang tidak dimiliki ketiganya: **algoritma**. Notebook 02 sudah
mendefinisikan kapan sebuah kesimpulan sah, tapi definisi bukan prosedur.
Membaca definisinya tidak memberi tahu caranya mencari jawaban, sama seperti
mengetahui arti "akar kuadrat" tidak otomatis memberi tahu caranya menghitung
akar kuadrat. Notebook 03 sudah menyediakan bahasanya, tapi bahasa tidak menalar
sendiri.

Notebook ini menutup lubang itu. Wumpus World dari Notebook 01 ditulis ulang
memakai bahasa Notebook 03, lalu pertanyaan "is there a pit in [1,2]" dijawab
dengan menjalankan definisi Notebook 02 apa adanya: daftar semua possible model,
buang yang membuat knowledge base salah, lalu periksa apakah masih ada model
tersisa yang membuat jawabannya salah. Kalau tidak ada satu pun, jawabannya
pasti. Itulah model checking.

Notebook ditutup dengan alasan kenapa cara selugas ini tidak bisa dipakai untuk
dunia yang cuma sedikit lebih besar.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 4.1 Symbols and Sentences in the Wumpus World

**Slide 32**

## Penjelasan

**Dari notebook sebelumnya.** Notebook 01 membangun Wumpus World sebagai
environment: gua 4 kali 4, pit yang mematikan, wumpus, dan lima sensor di slide
12. Cara kerja sensornya lokal dan sederhana, yaitu agent merasakan breeze kalau
kotak tetangganya berisi pit, dan stench kalau wumpus ada di dekatnya. Semua itu
masih berupa cerita dan simulator Python biasa; tidak ada satu pun kalimat logika
di dalamnya. Notebook 03 membangun sesuatu yang terpisah sama sekali:
proposition symbol, yaitu simbol yang cuma bisa bernilai benar atau salah, dan
logical connective untuk merangkainya jadi complex sentence yang nilainya bisa
dihitung lewat truth table.

**Yang dikerjakan di sub-topik ini.** Kedua barang itu disatukan: cerita Notebook
01 ditulis ulang memakai bahasa Notebook 03. Cara kerjanya, tiap fakta tentang
dunia diberi satu proposition symbol sendiri, lalu hubungan antar fakta ditulis
sebagai complex sentence. Hubungan yang di Notebook 01 masih berupa kalimat
bahasa Indonesia tentang cara kerja sensor, di sini berubah jadi rumus. Hasilnya
lima kalimat, dan setelah itu Wumpus World tidak lagi butuh gambar maupun
simulator untuk ditanyai.

Langkah pertama menentukan proposition symbol-nya. Ingat dari Notebook 03 bahwa
logika proposisional tidak punya variabel, jadi tidak ada cara menulis "kotak
[x, y]" sekali untuk semua x dan y. Setiap kotak harus punya simbolnya sendiri:

- $P_{x,y}$ bernilai benar kalau ada pit (lubang) di [x, y]
- $W_{x,y}$ bernilai benar kalau ada wumpus (monster) di [x, y], hidup maupun mati
- $B_{x,y}$ bernilai benar kalau agent merasakan breeze (angin) di [x, y]
- $S_{x,y}$ bernilai benar kalau agent merasakan stench (bau) di [x, y]

Mulai sekarang istilahnya dipakai dalam bahasa Inggris saja, mengikuti slide.

Perhatikan bahwa $B_{x,y}$ berbicara tentang apa yang **dirasakan** agent, bukan
tentang isi kotak. Membedakan keduanya penting, karena justru hubungan antara
percept dan isi kotak itulah yang nanti dipakai untuk menalar.

Untuk pertanyaan yang mau dijawab notebook ini, yaitu "is there a pit in [1,2]",
lima kalimat berikut sudah cukup.

![Slide 32](img/slide-32-formalisasi-wumpus.png)

- $R_1: \neg P_{1,1}$, there is no pit in [1,1]
- $R_2: B_{1,1} \Leftrightarrow (P_{1,2} \lor P_{2,1})$
- $R_3: B_{2,1} \Leftrightarrow (P_{1,1} \lor P_{2,2} \lor P_{3,1})$
- $R_4: \neg B_{1,1}$
- $R_5: B_{2,1}$

$R$ di situ bukan jenis simbol yang kelima. Berbeda dengan $P$, $W$, $B$, dan
$S$ yang merupakan proposition symbol dan punya nilai benar atau salah, $R_1$
sampai $R_5$ cuma label yang ditempelkan pada kalimatnya supaya gampang dirujuk.
Menulis "lihat $R_4$" jauh lebih enak daripada "lihat kalimat yang menyatakan
tidak ada breeze di [1,1]". Namanya diambil dari AIMA, dibaca sebagai *rule*.

Karena cuma label, $R_1$ sampai $R_5$ berada di luar logikanya dan tidak ikut
dihitung sebagai simbol. Itu sebabnya sub-topik 4.2 nanti menemukan tujuh
simbol, bukan dua belas. Kalau kelimanya diganti nama jadi $A$ sampai $E$,
tidak ada satu pun hasil di notebook ini yang berubah.

Kelima kalimat itu tidak sederajat, dan slide sengaja memisahkannya jadi dua
kelompok:

- $R_1$ sampai $R_3$ berlaku di **semua** wumpus world. Ini aturan mainnya,
  sudah benar sebelum agent melangkah sekali pun.
- $R_4$ dan $R_5$ hanya berlaku untuk world yang sedang dijelajahi agent
  sekarang. Ini hasil percept setelah agent berdiri di [1,1] lalu pindah ke
  [2,1], persis situasi Figure 7.3(b) di Notebook 01.

Kalau agent dipindah ke gua lain, $R_1$ sampai $R_3$ tetap dipakai dan cuma
$R_4$ dan $R_5$ yang diganti. Pemisahan inilah yang membuat knowledge base bisa
dipakai ulang, bukan ditulis ulang dari nol tiap kali.

### Why a Biconditional and Not a Plain Implication

Godaan pertama biasanya menulis $B_{1,1} \Rightarrow (P_{1,2} \lor P_{2,1})$:
kalau ada breeze, berarti ada pit di kotak tetangga. Arah ini memang yang
dipakai waktu membaca $R_5$. Karena $B_{2,1}$ benar, pasti ada pit di salah satu
kotak tetangga [2,1].

Tapi separuh pekerjaan agent justru butuh arah sebaliknya. Dari $R_4$, yaitu
$\neg B_{1,1}$, agent ingin menyimpulkan bahwa [1,2] dan [2,1] aman. Kesimpulan
itu hanya bisa ditarik kalau kita punya
$(P_{1,2} \lor P_{2,1}) \Rightarrow B_{1,1}$: kalau ada pit di kotak tetangga,
pasti ada breeze. Kontrapositifnya, tidak ada breeze berarti tidak ada pit di
kotak tetangga mana pun.

Dengan satu arah saja, salah satu dari dua kemampuan itu hilang:

| Kalau cuma ditulis | Masih bisa | Hilang |
|---|---|---|
| $B_{1,1} \Rightarrow (P_{1,2} \lor P_{2,1})$ | dari ada breeze: ada pit di dekat sini | dari tidak ada breeze: tidak bisa menyimpulkan apa-apa |
| $(P_{1,2} \lor P_{2,1}) \Rightarrow B_{1,1}$ | dari tidak ada breeze: kedua kotak tetangga aman | dari ada breeze: tidak bisa menyimpulkan apa-apa |

Kehilangan arah yang kedua fatal, karena berarti agent tidak akan pernah bisa
menyatakan sebuah kotak aman, dan permainannya jadi tidak bisa dimainkan. Karena
kedua arah sama-sama dibutuhkan, keduanya ditulis sekaligus lalu diringkas jadi
biconditional.

### Why the Breeze Rules Are Written Per Square

Kalimat "agent merasakan breeze di sebuah kotak jika dan hanya jika ada pit di
salah satu kotak tetangganya" adalah satu kalimat dalam bahasa Indonesia, tapi
bukan satu kalimat dalam logika proposisional. Karena tidak ada variabel,
kalimat itu harus disalin ulang untuk tiap kotak, dengan daftar tetangga yang
berbeda-beda. Grid 4 kali 4 berarti 16 salinan hanya untuk breeze, ditambah 16
lagi untuk stench.

Di sini cuma dua yang ditulis, yaitu untuk [1,1] dan [2,1], karena hanya dua
kotak itu yang nilai breeze-nya sudah diketahui agent. Kalimat untuk kotak lain
tidak salah, cuma tidak menyumbang apa pun terhadap pertanyaan yang sedang
ditanyakan, sementara tiap proposition symbol baru yang ikut masuk menggandakan
jumlah possible model yang harus diperiksa. Sub-topik 4.4 akan menunjukkan
berapa mahal ongkosnya.

Keterbatasan "harus disalin per kotak" ini bukan kelemahan kecil. Justru inilah
alasan utama kenapa logika proposisional akhirnya ditinggalkan untuk masalah
yang lebih besar.

## Contoh penerapan

Kelas `PropKB` di `logic.py` adalah knowledge base untuk logika proposisional.
Isinya satu atribut, `clauses`, berupa daftar clause, dan empat method:

| Method | Kegunaan |
|---|---|
| `tell(sentence)` | menambahkan sebuah kalimat ke KB |
| `ask_generator(query)` | menghasilkan `{}` kalau KB meng-entail query, dan tidak menghasilkan apa pun kalau tidak |
| `ask_if_true(query)` | pembungkus `ask_generator` yang langsung mengembalikan `True` atau `False` |
| `retract(sentence)` | menghapus clause yang berasal dari sebuah kalimat |

Dua method pertama itu persis operasi TELL dan ASK yang diperkenalkan Notebook
01 lewat slide 5. Di sana keduanya masih berupa kotak dalam diagram; di sini
keduanya fungsi yang benar-benar bisa dipanggil.

Bentuk `ask_generator` yang terasa aneh itu warisan dari knowledge base
first-order, di mana jawaban sebuah query bukan benar atau salah melainkan
substitusi variabel, misalnya `{x: 3}`. Di logika proposisional tidak ada
variabel yang perlu disubstitusi, jadi yang tersisa cuma substitusi kosong `{}`
sebagai tanda "iya, entailed". Untuk pemakaian sehari-hari, pakai `ask_if_true`.

In [ ]:
# Proposition symbols for the part of the cave the agent knows about.
# Pxy is true when square [x, y] holds a pit, Bxy when the agent feels a breeze
# while standing on it.
P11, P12, P21, P22, P31 = expr("P11, P12, P21, P22, P31")
B11, B21 = expr("B11, B21")

# R1-R3 hold in every wumpus world: these are the rules of the game.
R1 = ~P11
R2 = B11 | "<=>" | (P12 | P21)
R3 = B21 | "<=>" | (P11 | P22 | P31)

# R4-R5 are percepts, true only in the particular world the agent is walking
# through right now.
R4 = ~B11
R5 = B21

rules = {"R1": R1, "R2": R2, "R3": R3, "R4": R4, "R5": R5}
for name, sentence in rules.items():
    print(f"{name} = {sentence}")

Operator biconditional ditulis `|"<=>"|`, mengikuti pola `|"==>"|` yang sudah
dipakai di Notebook 03. Tanda kutipnya wajib, karena Python tidak punya operator
`<=>` sendiri dan `Expr` menumpang lewat operator `|`.

Yang tercetak sudah persis kelima kalimat di slide, cuma dengan tanda kurung
tambahan yang dipasang Python waktu mengurai ekspresinya.

In [ ]:
kb = PropKB()
for sentence in rules.values():
    kb.tell(sentence)

print("Clauses:", len(kb.clauses))
for clause in kb.clauses:
    print("   ", clause)

Lima kalimat masuk, sepuluh clause keluar. `tell` tidak menyimpan kalimat apa
adanya: dia mengubahnya dulu ke **conjunctive normal form**, yaitu konjungsi
dari disjungsi, lalu menyimpan tiap disjungsinya sebagai satu clause. Bentuk
seragam inilah yang nanti dipakai algoritma minggu depan.

$R_1$, $R_4$, dan $R_5$ masing-masing sudah berbentuk clause, jadi lewat begitu
saja. Yang memekar jadi banyak clause adalah kedua biconditional-nya.

In [ ]:
print("R2 as written :", R2)
print("R2 in CNF     :", to_cnf(R2))
print()
for clause in conjuncts(to_cnf(R2)):
    print("   clause:", clause)

Langkah pecahnya, karena ini bekal untuk materi minggu depan:

1. Biconditional dipecah jadi dua implication:
   $(B_{1,1} \Rightarrow (P_{1,2} \lor P_{2,1})) \land ((P_{1,2} \lor P_{2,1}) \Rightarrow B_{1,1})$
2. Tiap implication $a \Rightarrow b$ diganti $\neg a \lor b$:
   $(\neg B_{1,1} \lor P_{1,2} \lor P_{2,1}) \land (\neg (P_{1,2} \lor P_{2,1}) \lor B_{1,1})$
3. De Morgan pada bagian kedua:
   $(\neg B_{1,1} \lor P_{1,2} \lor P_{2,1}) \land ((\neg P_{1,2} \land \neg P_{2,1}) \lor B_{1,1})$
4. Distribusi $\lor$ terhadap $\land$:
   $(\neg B_{1,1} \lor P_{1,2} \lor P_{2,1}) \land (\neg P_{1,2} \lor B_{1,1}) \land (\neg P_{2,1} \lor B_{1,1})$

Tiga clause, sama dengan yang dicetak kode di atas. Dua clause terakhir itulah
yang tadi disebut arah kedua: keduanya menyatakan bahwa pit di [1,2] atau di
[2,1] memaksa $B_{1,1}$ jadi benar, sehingga dari $\neg B_{1,1}$ agent boleh
mencoret keduanya. Dengan cara yang sama $R_3$ pecah jadi empat clause, karena
kotak tetangga [2,1] ada tiga.

In [ ]:
query = ~P12

# ask_generator yields substitutions. For propositional logic the only possible
# substitution is the empty one.
print("ask_generator(~P12) ->", list(kb.ask_generator(query)))
print("ask_generator( P22) ->", list(kb.ask_generator(P22)))
print("ask_if_true  (~P12) ->", kb.ask_if_true(query))
print()

# retract removes the clauses a sentence contributed, so the same KB object can
# be reused for a what-if question.
kb.retract(R4)
print("without R4 -> clauses:", len(kb.clauses), "| KB entails ~P12:", kb.ask_if_true(~P12))
kb.tell(R4)
print("with    R4 -> clauses:", len(kb.clauses), "| KB entails ~P12:", kb.ask_if_true(~P12))

Baris pertama menghasilkan `[{}]`, satu substitusi kosong, artinya entailed.
Baris kedua menghasilkan `[]`, kosong sama sekali, artinya tidak entailed.
`ask_if_true` cuma memeriksa apakah generatornya menghasilkan sesuatu.

Dua baris terakhir menunjukkan siapa yang sebenarnya bekerja. Begitu $R_4$
dicabut, agent kehilangan fakta bahwa tidak ada breeze di [1,1], dan
$\neg P_{1,2}$ langsung berhenti entailed. Jadi kesimpulan "there is no pit in
[1,2]" lahir dari $R_2$ dan $R_4$ bersama-sama: aturan mainnya saja tidak cukup,
percept-nya saja juga tidak cukup.

---
# 4.2 Number of Possible Models

**Slide 33**

## Penjelasan

**Dari notebook sebelumnya.** Notebook 02 mendefinisikan entailment:
$KB \vDash \alpha$ berarti $\alpha$ benar di setiap model yang membuat KB benar,
atau dalam notasi slide 17, $M(KB) \subseteq M(\alpha)$. Cara kerjanya bukan
menghitung, melainkan menyaring. Dari semua dunia yang bisa dibayangkan, buang
dulu yang bertabrakan dengan isi KB, lalu lihat apakah $\alpha$ selamat di
seluruh sisanya. Notebook 02 juga sudah menyebut nama algoritma yang mengerjakan
penyaringan itu, yaitu model checking, tapi baru sebatas nama. Notebook 03
melengkapinya dengan pengertian model yang konkret: satu model adalah pemberian
nilai benar atau salah untuk setiap proposition symbol, seperti $m_1$ di slide
28.

**Yang dikerjakan di sub-topik ini.** Penyaringan tadi dijalankan, bukan cuma
didefinisikan. Sekarang bisa, karena dua syaratnya sudah lengkap: KB dari
sub-topik 4.1 sudah berbentuk kalimat formal, dan model sudah berbentuk `dict`
yang bisa dibangkitkan mesin. Cara kerjanya jadi sepele, yaitu perulangan
biasa atas seluruh model. Yang perlu diketahui lebih dulu cuma satu angka: ada
berapa model yang harus disaring.

Slide 33 menuliskan tujuan dan algoritmanya seperti ini:

> **Goal:** decide whether $KB \vDash \alpha$ for some sentence $\alpha$. For
> example, is $\neg P_{1,2}$ entailed by the Wumpus World KB?
>
> **Algorithm:** model checking, enumerate the models, and check that $\alpha$
> is true in every model in which KB is true.

Tidak ada trik di dalamnya. Model checking mengerjakan definisi Notebook 02 apa
adanya: daftar semua possible model, saring yang membuat KB benar, lalu periksa
$\alpha$ di sisanya. Karena mengikuti definisinya persis, algoritma ini otomatis
sound dan complete untuk logika proposisional.

Yang perlu dipikirkan justru ongkosnya, dan di situ slide sengaja meninggalkan
dua isian kosong:

> Currently there are 7 symbols: **?**
>
> Number of possible models: **?**

Sebelum lanjut, coba jawab sendiri dulu. Sebutkan ketujuh proposition symbol
yang muncul di $R_1$ sampai $R_5$, lalu hitung ada berapa kombinasi truth value
yang bisa dibentuk dari simbol sebanyak itu. Baru setelah punya jawaban sendiri,
jalankan kode di bawah untuk mengeceknya.

## Contoh penerapan

`prop_symbols()` mengumpulkan semua proposition symbol yang muncul di sebuah
kalimat, dan `associate("&", ...)` menggabungkan daftar clause jadi satu kalimat
konjungsi supaya bisa diperiksa sekaligus.

In [ ]:
kb_sentence = associate("&", kb.clauses)
symbols = sorted(prop_symbols(kb_sentence), key=str)

print("Symbols :", symbols)
print("Count   :", len(symbols))
print("Models  : 2 **", len(symbols), "=", 2 ** len(symbols))

Kedua isian di slide sekarang terjawab: **7 symbols** dan **128 possible
models**.

Ketujuh simbol itu adalah dua simbol breeze, $B_{1,1}$ dan $B_{2,1}$, ditambah
lima simbol pit, $P_{1,1}$, $P_{1,2}$, $P_{2,1}$, $P_{2,2}$, dan $P_{3,1}$. Tiap
simbol bisa bernilai benar atau salah secara bebas, jadi jumlah kombinasinya
$2^7 = 128$. Perhatikan bahwa angka 128 ini tidak ada hubungannya dengan isi
kalimatnya. Yang menentukan cuma banyaknya proposition symbol.

Implementasinya di `logic.py` ada di dua fungsi. `tt_entails` menyiapkan daftar
simbol, lalu `tt_check_all` menelusuri semuanya secara rekursif.

In [ ]:
psource(tt_entails)

In [ ]:
psource(tt_check_all)

Alurnya begini. `tt_check_all` mengambil satu proposition symbol dari daftar,
mencoba memberinya nilai `True`, lalu memanggil dirinya sendiri untuk sisa
simbolnya; setelah itu diulang dengan nilai `False`. Model dibangun sedikit demi
sedikit di parameter `model` sambil rekursi turun. Begitu daftar simbolnya
habis, model sudah lengkap dan barulah diperiksa: kalau KB salah di model itu,
model tersebut tidak relevan dan langsung dianggap lolos; kalau KB benar, nilai
$\alpha$ di sana yang menentukan. Hasil akhirnya digabung dengan `and`, jadi
cukup satu model tandingan untuk membuat seluruh jawaban `False`.

Sebelum dipakai ke Wumpus World, coba dulu di contoh yang cukup kecil untuk
diperiksa dengan mata.

In [ ]:
P, Q = expr("P, Q")

print("tt_entails(P & Q, Q) =", tt_entails(P & Q, Q))
print("tt_entails(P | Q, Q) =", tt_entails(P | Q, Q))

Yang pertama `True`: kalau diketahui P dan Q dua-duanya benar, tentu saja Q
benar. Yang kedua `False`, dan alasannya lebih menarik. Dengan dua proposition
symbol ada $2^2 = 4$ possible model, jadi truth table-nya cukup pendek untuk
dilihat seluruhnya.

In [ ]:
rows = []
for p_value, q_value in itertools.product([False, True], repeat=2):
    model = {P: p_value, Q: q_value}
    rows.append({
        "P": p_value,
        "Q": q_value,
        "KB = P | Q": pl_true(P | Q, model),
        "alpha = Q": pl_true(Q, model),
    })

pd.DataFrame(rows)

Baris yang membatalkan entailment adalah baris ketiga, yaitu P benar dan Q
salah. Di baris itu KB bernilai benar tapi $\alpha$ bernilai salah, jadi ada
possible world yang konsisten dengan pengetahuan kita tapi membuat kesimpulan
yang mau ditarik jadi salah. Satu baris seperti ini sudah cukup untuk
menjatuhkan seluruh klaim, dan itu persis yang dilakukan `and` di
`tt_check_all`.

Dua baris lain yang KB-nya benar, yaitu baris kedua dan keempat, sebenarnya
mendukung $\alpha$. Tapi entailment bukan soal mayoritas: satu tandingan saja
sudah membatalkan.

---
# 4.3 Rebuilding Figure 7.9

**Slide 34, Figure 7.9**

## Penjelasan

**Dari notebook sebelumnya.** Notebook 02 sudah menjawab pertanyaan yang sama
persis dengan yang akan dijawab di sini, tapi dengan tangan. Tiga kotak yang
belum diketahui digambar sebagai delapan possible models, ditandai mana yang
membuat KB benar, lalu terbaca langsung bahwa $\alpha_1$ entailed sedangkan
$\alpha_2$ tidak. Cara kerjanya mengandalkan mata, dan itu masih sanggup karena
delapan gambar muat di satu halaman. Notebook 03 menyediakan alat yang jauh
lebih sabar untuk pekerjaan sejenis, yaitu truth table, yang mendaftar seluruh
kombinasi nilai simbol secara sistematis lalu menghitung nilai kalimatnya baris
per baris.

**Yang dikerjakan di sub-topik ini.** Delapan gambar Notebook 02 diganti truth
table Notebook 03, dijalankan di atas KB formal sub-topik 4.1. Cara kerjanya
sama saja dengan yang manual, cuma pelakunya berganti: mesin yang mendaftar
kombinasinya dan mesin yang menandai baris mana yang membuat KB benar. Hasilnya
Figure 7.9, tabel 128 baris tanpa satu pun gambar.

Sub-topik 4.2 sudah menghitung angkanya, yaitu 7 symbols dan 128 possible
models. Figure 7.9 adalah 128 model itu ditulis lengkap.

![Figure 7.9](img/fig-7-9-truth-table-128-baris.png)

Tujuh kolom pertama adalah proposition symbol-nya, yaitu $B_{1,1}$, $B_{2,1}$,
$P_{1,1}$, $P_{1,2}$, $P_{2,1}$, $P_{2,2}$, dan $P_{3,1}$. Ketujuh kolom itulah
yang dienumerasi: tiap baris adalah satu possible world. Lima kolom berikutnya
adalah nilai $R_1$ sampai $R_5$ di model tersebut, dan kolom terakhir adalah KB,
yang bernilai benar hanya kalau kelima kalimatnya benar sekaligus.

Tiga hal yang perlu dibaca dari truth table itu:

- KB bernilai benar di tepat **3 dari 128 baris**. Sisanya tersingkir karena
  bertabrakan dengan salah satu kalimat, misalnya baris yang menyatakan ada pit
  di [1,1] langsung gugur oleh $R_1$.
- Di ketiga baris itu $P_{1,2}$ **selalu salah**. Tidak ada satu pun possible
  world yang konsisten dengan pengetahuan agent tapi punya pit di [1,2]. Karena
  itu $KB \vDash \neg P_{1,2}$, dan agent boleh melangkah ke sana.
- Di ketiga baris itu $P_{2,2}$ **kadang benar kadang salah**. Ada possible
  world yang konsisten dan ada pit-nya di [2,2], ada juga yang tidak. Jadi isi
  [2,2] belum bisa disimpulkan sama sekali.

Kalau kesimpulannya sama dengan Notebook 02, kenapa di sana cukup 8 gambar
sedangkan di sini 128 baris? Karena yang dienumerasi berbeda. Di sana yang
divariasikan cuma tiga kotak yang belum diketahui, jadi $2^3 = 8$. Di sini yang
dienumerasi adalah proposition symbol-nya, termasuk simbol breeze yang di
Notebook 02 belum punya simbol sendiri, jadi $2^7 = 128$. Possible model-nya
jauh lebih banyak, tapi jawabannya tidak berubah, karena $R_2$ sampai $R_5$
langsung mengunci nilai breeze-nya begitu isi kotaknya ditentukan.

## Contoh penerapan

Truth table-nya dibangun ulang dari nol, bukan disalin dari buku. Urutan
kolomnya disamakan dengan Figure 7.9 supaya hasilnya bisa dicocokkan baris per
baris.

In [ ]:
# Column order follows Figure 7.9 exactly.
figure_symbols = [B11, B21, P11, P12, P21, P22, P31]

rows = []
for values in itertools.product([False, True], repeat=len(figure_symbols)):
    model = dict(zip(figure_symbols, values))
    row = {str(symbol): value for symbol, value in model.items()}
    for name, sentence in rules.items():
        row[name] = pl_true(sentence, model)
    row["KB"] = all(row[name] for name in rules)
    rows.append(row)

table = pd.DataFrame(rows)
print("Rows:", len(table))
table.head()

128 baris, sesuai hitungan $2^7$. Lima baris pertama sudah memperlihatkan pola
enumerasinya: dari ketujuh kolom proposition symbol, yang paling kanan, yaitu
$P_{3,1}$, berubah paling cepat, persis seperti mencacah bilangan biner. Kolom
$R_1$ sampai KB di sebelah kanannya bukan simbol, melainkan hasil hitungan atas
baris tersebut. Sekarang ketiga klaim di atas diperiksa satu per satu.

In [ ]:
kb_rows = table[table["KB"]]

print("Total rows           :", len(table))
print("Rows where KB is true:", len(kb_rows))
print("P12 values there     :", sorted(kb_rows["P12"].unique()))
print("P22 values there     :", sorted(kb_rows["P22"].unique()))

Ketiganya cocok dengan keterangan di bawah Figure 7.9: 3 baris dari 128,
$P_{1,2}$ hanya pernah bernilai `False`, sedangkan $P_{2,2}$ pernah `False` dan
pernah `True`. Ketiga baris itu bisa dilihat langsung.

In [ ]:
kb_rows

Ini tiga baris yang di buku diberi garis bawah pada kolom KB. Nomor indeksnya
berurutan, 33 sampai 35, karena ketiganya cuma berbeda pada dua kolom simbol
paling kanan, dan kolom itulah yang berubah paling cepat waktu dienumerasi.

Bacaannya: ketiganya sepakat $B_{1,1}$ salah, $B_{2,1}$ benar, dan tidak ada pit
di [1,1], [1,2], maupun [2,1]. Yang mereka perselisihkan cuma [2,2] dan [3,1],
dan perselisihan itu wajar, karena $R_3$ dan $R_5$ hanya menuntut ada pit di
salah satu kotak tetangga [2,1], tanpa memberi tahu yang mana. Tiga baris itu
adalah tiga cara memenuhi tuntutan tersebut: pit di [3,1] saja, di [2,2] saja,
atau di kedua-duanya.

Hasil yang sama bisa didapat tanpa membuat truth table-nya sendiri, karena
`ask_if_true` sudah menjalankan enumerasi ini di belakang layar.

In [ ]:
print("KB entails ~P12 :", kb.ask_if_true(~P12))
print("KB entails ~P22 :", kb.ask_if_true(~P22))
print("KB entails  P22 :", kb.ask_if_true(P22))

Baris pertama menjawab pertanyaan utama sub-topik ini: there is no pit in [1,2],
terbukti.

Dua baris terakhir yang sering bikin kaget. `~P22` dan `P22` sama-sama
mengembalikan `False`, padahal keduanya kebalikan satu sama lain. Ini bukan bug.
`ask_if_true(X)` menanyakan apakah X benar di **setiap** model yang membuat KB
benar, bukan apakah X mungkin benar. Karena $P_{2,2}$ benar di dua baris dan
salah di satu baris, keduanya sama-sama punya model tandingan, jadi keduanya
sama-sama gagal.

Yang perlu diperbaiki bukan programnya, tapi cara membaca `False`. `False` di
sini berarti "tidak terbukti", bukan "terbukti tidak". Ini persis pembahasan
$\alpha_2$ di Notebook 02: sebuah kalimat bisa saja tidak entailed tanpa
negasinya jadi entailed. Kalau dicek untuk seluruh kotak, polanya kelihatan.

In [ ]:
for square, pit in [("[1,2]", P12), ("[2,1]", P21), ("[2,2]", P22), ("[3,1]", P31)]:
    no_pit = kb.ask_if_true(~pit)
    has_pit = kb.ask_if_true(pit)
    print(f"{square}  KB entails no pit: {str(no_pit):5s}  KB entails pit: {has_pit}")

Cuma [1,2] dan [2,1] yang statusnya sudah pasti, dan keduanya pasti aman. [2,2]
dan [3,1] sama-sama menjawab `False` dua kali, artinya belum diketahui. Bagi
agent, tiga status ini harus dibedakan dengan tegas: terbukti aman, terbukti
berbahaya, dan belum diketahui. Melangkah ke kotak yang belum diketahui adalah
taruhan, bukan kesimpulan. Latihan Soal 1 membahas ini lebih lanjut.

---
# 4.4 The Limits of Model Checking

**Slide 36**

## Penjelasan

**Dari notebook sebelumnya.** Notebook 02 memberi dua ukuran untuk menilai
sebuah algoritma inference, dan slide 22 merumuskannya begini:

> An inference algorithm that derives only entailed sentences is called sound or
> truth preserving.
>
> An inference algorithm is complete if it can derive any sentence that is
> entailed.

Cara kerja kedua ukuran itu sama, yaitu membandingkan hasil algoritma dengan
entailment yang sudah didefinisikan lebih dulu. Sound berarti algoritmanya tidak
mengarang: semua yang diturunkannya memang benar-benar entailed. Complete
berarti algoritmanya tidak kecolongan: semua yang entailed pasti bisa
diturunkannya.

**Yang dikerjakan di sub-topik ini.** Model checking diukur dengan dua ukuran
itu, lalu ditanya satu hal yang belum pernah ditanyakan di notebook mana pun:
berapa ongkosnya. Sound dan complete sama sekali tidak peduli algoritmanya
selesai dalam sedetik atau seribu tahun, dan justru di celah itulah kelemahan
model checking bersembunyi.

Sebelum ke sana, ini ringkasan perjalanan keempat notebook:

| Konsep | Diperkenalkan di | Diwakili oleh |
|---|---|---|
| Knowledge base, TELL dan ASK | 01 | `PropKB`, `tell`, `ask_if_true` |
| Model dan possible world | 02 | `dict` dari proposition symbol ke `True` atau `False` |
| Entailment $KB \vDash \alpha$ | 02 | `tt_entails(kb, alpha)` |
| Sound dan complete | 02 | sifat algoritmanya, bukan fungsi tersendiri |
| Proposition symbol dan logical connectives | 03 | `expr`, `Expr`, operator Python untuk connectives |
| Truth value di satu model | 03 | `pl_true(sentence, model)` |
| Truth table | 03 | enumerasi dengan `itertools.product` |
| Model checking | 04 | `tt_entails`, `tt_check_all` |
| Conjunctive normal form | 04, sekilas | `to_cnf`, `conjuncts` |

Diukur dengan dua ukuran Notebook 02, model checking memenuhi dua-duanya. Bukan
karena dirancang dengan pintar, tapi karena dia memang definisi entailment yang
dieksekusi apa adanya, sehingga tidak ada ruang untuk mengarang maupun
kecolongan. Dia tidak butuh kepintaran tambahan apa pun.

Masalahnya di ongkos. Jumlah possible model tumbuh $2^n$ terhadap jumlah
proposition symbol, dan pertumbuhan itu tidak peduli seberapa sederhana
kalimatnya. KB di sub-topik 4.1 cuma butuh 128 model karena simbolnya tujuh,
tapi ketujuh simbol itu menyentuh lima kotak dan cuma memodelkan pit-nya,
ditambah breeze di dua kotak. Begitu wumpus dan stench ikut dimodelkan, jumlah
simbolnya melonjak. KB pada Latihan Soal 2 nanti sudah memakai 18 simbol, dan
itu pun baru mencakup enam kotak dari enam belas.

## Contoh penerapan

Tidak ada konsep baru di sini, cuma perhitungan kecil untuk melihat kurvanya.
Anggap satu model bisa dievaluasi dalam 1 mikrodetik, yang sudah sangat optimis
untuk Python.

In [ ]:
def human_time(seconds):
    # Report a duration in the largest unit that still reads sensibly.
    units = [
        (60 * 60 * 24 * 365.25, "years"),
        (60 * 60 * 24, "days"),
        (60 * 60, "hours"),
        (60, "minutes"),
        (1, "seconds"),
    ]
    for size, name in units:
        if seconds >= size:
            return f"{seconds / size:,.1f} {name}"
    return f"{seconds * 1000:,.1f} milliseconds"


cases = [
    ("KB in section 4.1", 7),
    ("KB in exercise 2", 18),
    ("Full 4 x 4 grid, P W B S on every square", 64),
]

for label, symbol_count in cases:
    models = 2 ** symbol_count
    seconds = models / 1_000_000  # one microsecond per model
    print(f"{label:42s} n = {symbol_count:2d}")
    print(f"{'':42s} models = {models:,}")
    print(f"{'':42s} at 1 us per model = {human_time(seconds)}")
    print()

Baris terakhir itu bukan lelucon. Grid 4 kali 4 dengan empat jenis simbol per
kotak sudah butuh 64 proposition symbol, dan memeriksa seluruh possible
model-nya makan ratusan ribu tahun. Padahal 64 simbol pun masih jauh dari
lengkap: belum ada simbol untuk posisi agent, arah hadapnya, panah yang belum
ditembakkan, emas, atau waktu.

Jadi model checking sound dan complete, tapi tidak bisa dipakai. Dan penyebabnya
bukan implementasi yang lambat. Menggandakan kecepatan komputer cuma menambah
satu proposition symbol pada batas yang sanggup dikerjakan.

Jalan keluarnya adalah berhenti memeriksa model satu per satu, dan mulai
memanipulasi kalimatnya secara langsung. Itu materi minggu depan seperti di
slide 36:

> **Next Week: Propositional Theorem Proving**
>
> Proof construction without consulting models.

Yang akan dibahas di sana, semuanya sudah tersedia di `logic.py` kalau ada yang
penasaran ingin mengintip duluan:

- konversi ke CNF lalu **resolution**, lewat `to_cnf` dan `pl_resolution`
- **Horn clause** dengan forward dan backward chaining, lewat `PropDefiniteKB`
  dan `pl_fc_entails`
- model checking yang jauh lebih efisien lewat `dpll_satisfiable` dan `WalkSAT`

Tiga pendekatan itu menjawab pertanyaan yang sama dengan `tt_entails`, tapi
tanpa harus mendaftar seluruh possible world lebih dulu.

---
# Latihan Soal

Empat soal, dikerjakan berurutan. Soal 2 adalah latihan resmi dari slide dan
porsinya paling besar; tiga soal lainnya menyiapkan dan melanjutkannya.

Tiap soal punya sel kosong untuk jawabanmu, lalu pembahasan yang bisa
dibuka-tutup. Kerjakan dulu sampai mentok sebelum membuka pembahasannya.

## Soal 1

**Tingkat pemahaman.**

Di sub-topik 4.3 terlihat bahwa `kb.ask_if_true(~P22)` dan `kb.ask_if_true(P22)`
sama-sama mengembalikan `False`.

a. Jelaskan kenapa keduanya `False`, dan kenapa itu bukan tanda ada yang salah
   di `logic.py`.
b. Apa artinya bagi agent yang harus memutuskan melangkah ke [2,2] atau tidak?

Jawab dengan kalimat sendiri di sel di bawah, boleh ditulis sebagai komentar.

In [ ]:
# Your answer here.

<details>
<summary>Klik untuk melihat pembahasan</summary>

**a.** `ask_if_true(X)` menanyakan apakah X benar di **setiap** model yang
membuat KB benar, bukan apakah X mungkin benar. Dari tiga baris yang lolos di
sub-topik 4.3, $P_{2,2}$ bernilai salah di satu baris dan benar di dua baris.
Akibatnya:

- $\neg P_{2,2}$ gagal, karena ada baris yang $P_{2,2}$-nya benar
- $P_{2,2}$ gagal, karena ada baris yang $P_{2,2}$-nya salah

Keduanya `False` justru karena KB memang belum menentukan isi [2,2]. Yang keliru
bukan programnya, melainkan harapan bahwa `False` berarti "tidak". `False` di
sini berarti "tidak terbukti", dan itu bisa berlaku untuk sebuah kalimat dan
negasinya sekaligus.

Bandingkan dengan `ask_if_true(~P12)` yang mengembalikan `True`. Di ketiga baris
itu $P_{1,2}$ selalu salah, jadi tidak ada model tandingan sama sekali.

**b.** Bagi agent, [2,2] berstatus belum diketahui, bukan aman dan bukan
berbahaya. Melangkah ke sana adalah taruhan, bukan kesimpulan. Yang bisa
dilakukan agent adalah pindah ke kotak yang sudah terbukti aman, atau
mengumpulkan percept baru dulu sampai [2,2] terjawab.

Soal 2 menunjukkan persis hal itu terjadi. Begitu agent melangkah ke [1,2] dan
mendapati tidak ada breeze di sana, $\neg P_{2,2}$ berubah dari `False` jadi
`True`.

Catatan: kalau yang ingin ditanyakan sebenarnya "mungkinkah ada pit di [2,2]",
maka pertanyaannya bukan entailment melainkan satisfiability, dan itu topik
minggu depan.

</details>

## Soal 2: 32 Possible Worlds

**Slide 35. Tingkat penerapan.**

Ini latihan resmi dari slide dan jadi soal utama notebook ini.

![Soal slide 35](img/soal-32-possible-worlds.png)

> Diketahui agent telah sampai pada kondisi seperti pada gambar: tidak ada
> apa-apa pada [1,1], ada input berupa bau (stench) pada [1,2], dan belum tahu
> isi dari [1,3], [2,2], dan [3,1]. Masing-masing dari posisi tersebut dapat
> berisi sebuah lubang (pit) dan maksimal satu posisi dapat berisi monster
> (Wumpus).
>
> Tentukan kondisi yang mungkin (sebanyak 32 possible worlds).
>
> Tandai kondisi di mana KB bernilai benar dan di mana masing-masing kalimat
> berikut benar:
>
> $\alpha_2$ = "There is no pit in [2,2]"
>
> $\alpha_3$ = "There is a wumpus in [1,3]"
>
> Dengan demikian tunjukkan bahwa $KB \vDash \alpha_2$ dan $KB \vDash \alpha_3$.

### Dari mana angka 32

Bentuknya sama persis dengan delapan possible models di Notebook 02, yaitu
daftar kondisi yang mungkin lalu tandai mana yang membuat KB benar. Bedanya
agent sudah maju satu langkah, dan kali ini wumpus ikut dihitung.

Yang divariasikan cuma tiga kotak yang belum diketahui, yaitu [1,3], [2,2], dan
[3,1]. Kotak yang sudah dikunjungi, yaitu [1,1], [2,1], dan [1,2], tidak ikut
divariasikan: agent pernah berdiri di sana dan masih hidup, jadi ketiganya pasti
tidak berisi pit maupun wumpus.

- Tiap kotak yang belum diketahui bisa berisi pit atau tidak: $2^3 = 8$
- Wumpus ada di salah satu dari ketiga kotak itu, atau tidak di ketiganya: 4

Totalnya $8 \times 4 = 32$ possible worlds.

### Percept lengkap dari gambar

Yang paling sering terlewat adalah percept negatifnya, terutama "tidak ada
stench di [2,1]". Justru itu yang nanti dipakai untuk menyimpulkan $\alpha_3$.

| Kotak | Breeze | Stench |
|---|---|---|
| [1,1] | tidak | tidak |
| [2,1] | ya | tidak |
| [1,2] | tidak | ya |

### Kalimat untuk stench

KB di sub-topik 4.1 cuma punya pit dan breeze. Untuk soal ini perlu simbol
$W_{x,y}$ dan $S_{x,y}$, plus kalimat untuk stench. Kalimatnya tidak ada di
slide 32, jadi diturunkan sendiri dari deskripsi Sensors di slide 12:

> In the square containing the wumpus and in the directly (not diagonally)
> adjacent squares, the agent will perceive a Stench.

Perhatikan "in the square containing the wumpus". Berbeda dengan breeze, kotak
wumpus itu sendiri juga mengeluarkan stench, jadi kotaknya ikut masuk ke ruas
kanan:

$$S_{1,2} \Leftrightarrow (W_{1,2} \lor W_{1,1} \lor W_{1,3} \lor W_{2,2})$$

Kalimat breeze tidak perlu perlakuan yang sama, karena agent tidak akan pernah
sempat merasakan apa pun dari dalam kotak yang berisi pit. Sama seperti breeze,
kalimat stench ditulis untuk tiap kotak yang sudah dikunjungi, yaitu [1,1],
[2,1], dan [1,2].

Daftar tetangganya: [1,1] bertetangga dengan [1,2] dan [2,1]; [2,1] bertetangga
dengan [1,1], [2,2], dan [3,1]; [1,2] bertetangga dengan [1,1], [1,3], dan
[2,2].

### Yang harus dikerjakan

Knowledge base versi lengkapnya sudah disiapkan di dua sel berikut, termasuk
jawaban mesinnya, supaya bisa dipakai sebagai pembanding. Tugasmu adalah
menunjukkan **kenapa** jawabannya begitu, dengan tangan:

a. Bangkitkan 32 possible worlds tersebut sebagai DataFrame, dengan kolom untuk
   pit di tiap kotak, posisi wumpus, KB, $\alpha_2$, dan $\alpha_3$.
b. Tampilkan baris yang KB-nya bernilai benar saja.
c. Periksa nilai $\alpha_2$ dan $\alpha_3$ di baris-baris itu, lalu tuliskan
   kesimpulannya.
d. Bandingkan hasilmu dengan jawaban `ask_if_true` di bawah.

In [ ]:
# Six squares are involved: three already visited, three still unknown.
P11, P12, P13, P21, P22, P31 = expr("P11, P12, P13, P21, P22, P31")
W11, W12, W13, W21, W22, W31 = expr("W11, W12, W13, W21, W22, W31")
B11, B12, B21 = expr("B11, B12, B21")
S11, S12, S21 = expr("S11, S12, S21")

kb2 = PropKB()

# The agent survived every square it stood on, so those hold neither pit nor wumpus.
for sentence in [~P11, ~W11, ~P21, ~W21, ~P12, ~W12]:
    kb2.tell(sentence)

# A square is breezy iff a neighbour holds a pit. A square smells iff it or one
# of its neighbours holds the wumpus. Written once per visited square, with the
# neighbours listed by hand.
kb2.tell(B11 | "<=>" | (P12 | P21))
kb2.tell(B21 | "<=>" | (P11 | P22 | P31))
kb2.tell(B12 | "<=>" | (P11 | P13 | P22))
kb2.tell(S11 | "<=>" | (W11 | W12 | W21))
kb2.tell(S21 | "<=>" | (W21 | W11 | W22 | W31))
kb2.tell(S12 | "<=>" | (W12 | W11 | W13 | W22))

# The exercise allows at most one wumpus among the three unknown squares.
kb2.tell(~(W13 & W22))
kb2.tell(~(W13 & W31))
kb2.tell(~(W22 & W31))

# Percepts read off the picture.
for sentence in [~B11, ~S11, B21, ~S21, S12, ~B12]:
    kb2.tell(sentence)

symbols2 = sorted(prop_symbols(associate("&", kb2.clauses)), key=str)
print("Clauses :", len(kb2.clauses))
print("Symbols :", len(symbols2), symbols2)
print("Models  : 2 **", len(symbols2), "=", 2 ** len(symbols2))

Delapan belas proposition symbol, jadi 262.144 possible model yang harus
ditelusuri untuk sekali bertanya.

Bandingkan dengan sub-topik 4.1 yang cuma punya 7 symbols dan 128 possible
models. Sebelas simbol tambahannya berasal dari tiga tempat: satu kotak baru
[1,3] yang perlu simbol pit-nya sendiri, satu simbol breeze untuk [1,2] yang
tadinya belum ikut, lalu enam simbol wumpus dan tiga simbol stench yang tadinya
belum dimodelkan sama sekali. Sebelas simbol itu saja sudah membuat jumlah
possible model-nya membengkak $2^{11} = 2.048$ kali lipat.

Perhatikan waktu yang tercetak sel berikut. Itu bukti langsung untuk sub-topik
4.4.

In [ ]:
%%time
print("KB entails alpha2 (~P22) :", kb2.ask_if_true(~P22))
print("KB entails alpha3  (W13) :", kb2.ask_if_true(W13))

Mesin sudah menjawab `True` untuk keduanya, sesuai yang diminta slide. Tapi
jawaban itu belum menjelaskan apa pun. Sekarang tunjukkan sendiri kenapa, dengan
mengenumerasi 32 possible worlds-nya.

Petunjuk: dalam enumerasi manual, breeze dan stench tidak perlu jadi variabel.
Begitu isi ketiga kotak ditentukan, nilai breeze dan stench di kotak yang sudah
dikunjungi langsung ikut tertentu. Itu sebabnya cukup 32 possible worlds, bukan
262.144 possible models.

In [ ]:
# (a) Build the 32 possible worlds.
# Your answer here.

In [ ]:
# (b) Keep only the worlds in which KB is true.
# Your answer here.

In [ ]:
# (c) Check alpha2 and alpha3 in those worlds, then state the conclusion.
# Your answer here.

In [ ]:
# (d) Compare with the ask_if_true answers printed above.
# Your answer here.

<details>
<summary>Klik untuk melihat pembahasan</summary>

**a dan b.** Yang dienumerasi cuma pit di tiga kotak dan posisi wumpus. Nilai
breeze dan stench dihitung dari situ memakai daftar tetangga di atas.

```python
unknown = ["13", "22", "31"]

rows = []
for pits in itertools.product([False, True], repeat=3):
    for wumpus in [*unknown, None]:
        pit = dict(zip(unknown, pits))

        # Percepts implied by this world. Every visited square is empty, so it
        # contributes nothing to any breeze or stench.
        b11 = False
        s11 = False
        b21 = pit["22"] or pit["31"]
        s21 = wumpus in ("22", "31")
        b12 = pit["13"] or pit["22"]
        s12 = wumpus in ("13", "22")

        rows.append({
            "P13": pit["13"],
            "P22": pit["22"],
            "P31": pit["31"],
            "wumpus": f"[{wumpus[0]},{wumpus[1]}]" if wumpus else "-",
            "KB": (not b11) and (not s11) and b21 and (not s21) and (not b12) and s12,
            "alpha2": not pit["22"],
            "alpha3": wumpus == "13",
        })

worlds = pd.DataFrame(rows)
print("Possible worlds:", len(worlds))
worlds[worlds["KB"]]
```

Hasilnya 32 baris, dan KB bernilai benar di **tepat satu** baris: tidak ada pit
di [1,3] dan [2,2], ada pit di [3,1], dan wumpus di [1,3].

**c.** Di satu-satunya baris itu, $\alpha_2$ dan $\alpha_3$ dua-duanya benar.

```python
kb_worlds = worlds[worlds["KB"]]
print("alpha2 true in every KB world :", bool(kb_worlds["alpha2"].all()))
print("alpha3 true in every KB world :", bool(kb_worlds["alpha3"].all()))
```

Karena tidak ada satu pun possible world yang membuat KB benar tapi $\alpha_2$
atau $\alpha_3$ salah, maka $KB \vDash \alpha_2$ dan $KB \vDash \alpha_3$.

Penalaran di balik baris itu, langkah demi langkah:

1. Agent tidak merasakan breeze di [1,2]. Tetangga [1,2] adalah [1,1], [1,3],
   dan [2,2], jadi ketiganya bebas pit. Ini langsung membuktikan $\alpha_2$.
2. Agent merasakan breeze di [2,1]. Tetangganya [1,1], [2,2], dan [3,1]. [1,1]
   sudah dikunjungi dan [2,2] baru saja dicoret, jadi pit-nya pasti di [3,1].
3. Agent tidak merasakan stench di [2,1], jadi [2,2] dan [3,1] bebas wumpus.
4. Agent merasakan stench di [1,2]. Tetangganya [1,1], [1,3], dan [2,2]. [1,1]
   sudah dikunjungi dan [2,2] baru dicoret di langkah 3, jadi wumpus pasti di
   [1,3]. Ini membuktikan $\alpha_3$.

Langkah 3 itu yang paling mudah terlewat. Tanpa percept "tidak ada stench di
[2,1]", [2,2] tetap jadi kandidat wumpus dan $\alpha_3$ tidak bisa disimpulkan.

**d.** Sama persis dengan jawaban `ask_if_true`. Bedanya cuma ongkos: enumerasi
manual memeriksa 32 possible worlds dan selesai seketika, sedangkan
`ask_if_true` memeriksa 262.144 possible models dan butuh waktu yang terasa.

Selisih itu bukan karena `tt_entails` ditulis dengan buruk. Enumerasi manual
tahu bahwa nilai breeze dan stench bisa dihitung dari isi kotak, sehingga tidak
perlu dijadikan variabel bebas. `tt_entails` tidak tahu apa-apa tentang Wumpus
World; yang dilihatnya cuma 18 proposition symbol yang semuanya harus dicoba
benar dan salah. Pengetahuan seperti itulah yang dimanfaatkan algoritma minggu
depan.

</details>

## Soal 3

**Tingkat analisis.**

Pakai `kb2` dari Soal 2. Selain $\alpha_2$ dan $\alpha_3$, kesimpulan apa lagi
yang bisa ditarik dari knowledge base itu?

Cari minimal dua kesimpulan tambahan, tuliskan dalam bahasa Indonesia dulu, baru
terjemahkan jadi query dan buktikan dengan kode. Untuk tiap kesimpulan,
jelaskan juga percept mana yang membuatnya bisa ditarik.

In [ ]:
# Your answer here.

<details>
<summary>Klik untuk melihat pembahasan</summary>

Ada beberapa, dan semuanya bisa dibaca dari satu-satunya possible world yang
membuat KB benar. Empat contoh:

```python
extra = {
    "There is a pit in [3,1]": P31,
    "There is no pit in [1,3]": ~P13,
    "There is no wumpus in [2,2]": ~W22,
    "[2,2] is safe: no pit and no wumpus": ~P22 & ~W22,
}

for description, query in extra.items():
    print(f"{description:38s} -> {kb2.ask_if_true(query)}")
```

Keempatnya `True`. Asal-usulnya:

- **There is a pit in [3,1].** Dari breeze di [2,1] ditambah [2,2] yang sudah
  dicoret oleh "tidak ada breeze di [1,2]". Ini kesimpulan positif pertama yang
  bisa ditarik agent tentang bahaya, dan cocok dengan label `P!` di gambar soal.
- **There is no pit in [1,3].** Dari "tidak ada breeze di [1,2]", karena [1,3]
  tetangganya.
- **There is no wumpus in [2,2].** Dari "tidak ada stench di [2,1]".
- **[2,2] aman sepenuhnya.** Gabungan dua kesimpulan di atas. Inilah yang di
  gambar soal ditandai `OK`, dan inilah jawaban atas Soal 1: kotak yang di
  sub-topik 4.3 masih berstatus belum diketahui, sekarang sudah terbukti aman
  setelah satu percept tambahan.

Perhatikan bahwa query pada baris terakhir berupa konjungsi. `ask_if_true`
menerima kalimat apa pun, tidak harus satu proposition symbol.

Yang menarik justru batasnya: coba tanyakan `kb2.ask_if_true(P13)`. Hasilnya
`False`, walaupun `~P13` bernilai `True`. Konsisten, karena kalau $\neg P_{1,3}$
entailed maka $P_{1,3}$ tentu tidak. Bandingkan dengan kasus [2,2] di Soal 1
yang dua-duanya `False`; di sini situasinya berbeda karena isi [1,3] memang
sudah terjawab.

</details>

## Soal 4

**Tingkat analisis.**

Sub-topik 4.4 menyebut angka 64 simbol tanpa menurunkannya. Turunkan sendiri.

a. Berapa banyak proposition symbol yang dibutuhkan kalau seluruh grid 4 kali 4
   dimodelkan lengkap dengan pit, wumpus, breeze, dan stench?
b. Berapa jumlah possible model-nya?
c. Kalau satu model butuh 1 mikrodetik untuk dievaluasi, berapa lama model
   checking-nya selesai? Nyatakan dalam satuan yang masuk akal.
d. Apa yang berubah kalau komputernya dibuat seribu kali lebih cepat?

In [ ]:
# Your answer here.

<details>
<summary>Klik untuk melihat pembahasan</summary>

```python
symbol_count = 16 * 4
models = 2 ** symbol_count
seconds = models / 1_000_000
years = seconds / (60 * 60 * 24 * 365.25)

print("Symbols :", symbol_count)
print("Models  :", f"{models:,}")
print("Seconds :", f"{seconds:,.0f}")
print("Years   :", f"{years:,.0f}")
```

**a.** Grid 4 kali 4 berisi 16 kotak, dan tiap kotak butuh empat proposition
symbol, yaitu $P_{x,y}$, $W_{x,y}$, $B_{x,y}$, dan $S_{x,y}$. Totalnya
$16 \times 4 = 64$ symbols.

**b.** $2^{64} = 18.446.744.073.709.551.616$ possible models.

**c.** Pada 1 mikrodetik per model, waktunya sekitar
$1{,}8 \times 10^{13}$ detik, atau kira-kira **584.542 tahun**.

**d.** Nyaris tidak ada yang berubah: 584.542 tahun dibagi seribu masih sekitar
585 tahun. Faktor konstan tidak berarti apa-apa melawan pertumbuhan $2^n$. Cara
lain melihatnya, komputer seribu kali lebih cepat cuma menambah sekitar 10
proposition symbol pada batas yang sanggup dikerjakan, karena
$2^{10} \approx 1000$.

Itu pun masih perkiraan yang terlalu murah hati. Wumpus World lengkap juga butuh
simbol untuk posisi agent, arah hadapnya, panah yang belum ditembakkan, dan
emas, dan sebagian di antaranya berubah tiap langkah waktu sehingga harus
disalin untuk tiap waktu t.

Kesimpulannya, model checking dengan truth table tidak bisa diselamatkan dengan
hardware. Yang harus diganti adalah pendekatannya, dan itu materi minggu depan.

</details>